In [1]:
from google.colab import files
uploaded = files.upload()  # upload Reviews.csv

Saving Reviews.csv to Reviews.csv


In [2]:
!pip install transformers datasets accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# NexCart Intelligence — DistilBERT Sentiment Analysis
**Dataset:** Amazon Fine Food Reviews (568K reviews)
**Model:** DistilBertForSequenceClassification (fine-tuned)
**Target:** F1 > 0.85 on validation set

In [3]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Libraries loaded ✅")

PyTorch version: 2.10.0+cu128
CUDA available: True
Libraries loaded ✅


## Step 1 — Load & Prepare Dataset

---

In [4]:
df = pd.read_csv('Reviews.csv')
print(f"Full dataset shape: {df.shape}")
print(df[['Score', 'Text']].head(3))

Full dataset shape: (568454, 10)
   Score                                               Text
0      5  I have bought several of the Vitality canned d...
1      1  Product arrived labeled as Jumbo Salted Peanut...
2      4  This is a confection that has been around a fe...


In [5]:
# Keep only Score and Text columns
df = df[['Score', 'Text']].dropna()

# Binary classification: 1-3 → negative (0), 4-5 → positive (1)
df['label'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nPositive: {df['label'].sum()} | Negative: {(df['label']==0).sum()}")

Label distribution:
label
1    443777
0    124677
Name: count, dtype: int64

Positive: 443777 | Negative: 124677


In [6]:
# Separate positive and negative
positive = df[df['label'] == 1]
negative = df[df['label'] == 0]

print(f"Total positive: {len(positive)}")
print(f"Total negative: {len(negative)}")

# Balance: 50K each
sample_size = min(50_000, len(positive), len(negative))
positive_sample = positive.sample(n=sample_size, random_state=42)
negative_sample = negative.sample(n=sample_size, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_sample, negative_sample])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"Label distribution:\n{df_balanced['label'].value_counts()}")

Total positive: 443777
Total negative: 124677

Balanced dataset shape: (100000, 3)
Label distribution:
label
0    50000
1    50000
Name: count, dtype: int64


In [7]:
# 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df_balanced, test_size=0.30, random_state=42, stratify=df_balanced['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f"Train size:      {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size:       {len(test_df)}")

Train size:      70000
Validation size: 15000
Test size:       15000


## Step 3 — Tokenization

In [8]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer loaded: {MODEL_NAME} ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased ✅


In [9]:
def tokenize(batch):
    return tokenizer(
        batch['Text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['Text', 'label']])
val_dataset   = Dataset.from_pandas(val_df[['Text', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['Text', 'label']])

# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete ✅")
print(f"Train dataset: {train_dataset}")

Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenization complete ✅
Train dataset: Dataset({
    features: ['Text', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 70000
})


## Step 4 — Fine-Tune DistilBERT

In [10]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f"Model loaded: {MODEL_NAME} ✅")
print(f"Parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased ✅
Parameters: 66,955,010


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    accuracy = (predictions == labels).mean()
    return {
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4)
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning... (this takes 20-40 mins on CPU)")
trainer.train()
print("Fine-tuning complete ✅")

Starting fine-tuning... (this takes 20-40 mins on CPU)


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.196889,0.216451,0.922400,0.922400
2,0.159980,0.218112,0.932400,0.932400
3,0.131853,0.285580,0.932800,0.932800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete ✅


## Step 5 — Evaluate on Validation Set

In [14]:
results = trainer.evaluate(val_dataset)

print(f"Validation F1:       {results['eval_f1']}")
print(f"Validation Accuracy: {results['eval_accuracy']}")
print(f"\nTarget F1 > 0.85: {'✅ PASSED' if results['eval_f1'] > 0.85 else '❌ FAILED'}")

Training Loss,Validation Loss,Epoch,F1,Accuracy
0.131853,0.285580,3,0.932800,0.932800


Validation F1:       0.9328
Validation Accuracy: 0.9328

Target F1 > 0.85: ✅ PASSED


In [15]:
# Get all predictions on test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(classification_report(
    true_labels,
    pred_labels,
    target_names=['negative', 'positive']
))

              precision    recall  f1-score   support

    negative       0.93      0.93      0.93      7500
    positive       0.93      0.93      0.93      7500

    accuracy                           0.93     15000
   macro avg       0.93      0.93      0.93     15000
weighted avg       0.93      0.93      0.93     15000



## Step 6 — Save Model + Tokenizer

In [22]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
import os

# Load best checkpoint
model = DistilBertForSequenceClassification.from_pretrained('/content/results/checkpoint-13125')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Save properly
os.makedirs('/content/saved_models/sentiment_model', exist_ok=True)
model.save_pretrained('/content/saved_models/sentiment_model')
tokenizer.save_pretrained('/content/saved_models/sentiment_model')

print(os.listdir('/content/saved_models/sentiment_model'))
print("Saved ✅")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']
Saved ✅


## Step 7 — Quick Inference Test

In [18]:
from transformers import pipeline

# Reload both from saved path to ensure they match
sentiment_pipeline = pipeline(
    "text-classification",
    model="../saved_models/sentiment_model",
    tokenizer="../saved_models/sentiment_model",
    device=-1  # CPU
)

# Test positive review
result1 = sentiment_pipeline("This product is absolutely amazing, best purchase ever!")
print(f"Positive test: {result1}")

# Test negative review
result2 = sentiment_pipeline("Terrible quality, complete waste of money, do not buy.")
print(f"Negative test: {result2}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Positive test: [{'label': 'LABEL_1', 'score': 0.9987321496009827}]
Negative test: [{'label': 'LABEL_0', 'score': 0.99936443567276}]


In [20]:
import os

# Check all possible locations
paths = [
    '/content/saved_models',
    '/content/results',
    '/content/sentiment_model',
    '/content'
]

for path in paths:
    if os.path.exists(path):
        print(f"Found: {path}")
        print(os.listdir(path))

Found: /content/results
['checkpoint-8750', 'checkpoint-4375', 'checkpoint-13125']
Found: /content
['.config', 'Reviews.csv', 'results', 'sample_data']


In [23]:
import shutil
from google.colab import files

shutil.make_archive('sentiment_model', 'zip', '/content/saved_models/sentiment_model')
files.download('sentiment_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>